In [ ]:
"""
Program Name: PDMtoWAV.py
Description: converts pdm to wav
Programmer(s): Sam Harrison, ChatGPT
Date Made: 4/22
Date(s) Revised:
Preconditions: 
Postconditions: 
Errors/Exceptions:
Side Effects:
Invariants: 
Known Faults:
"""

import numpy as np
from scipy.signal import lfilter, firwin
import wave

def PDMtoWAV(pdm_file):
    # Configuration
    pdm_file = pdm_file  # raw binary 1-bit file
    wav_file = 'output.wav'
    pdm_sampling_rate = 3000000  # 3 MHz (adjust as needed)
    wav_sampling_rate = 48000    # desired output rate
    decimation_factor = int(pdm_sampling_rate / wav_sampling_rate)

    # Step 1: Read raw PDM data
    with open(pdm_file, 'rb') as f:
        raw_bytes = np.frombuffer(f.read(), dtype=np.uint8)

    # Unpack bits: 8 samples per byte
    pdm_data = np.unpackbits(raw_bytes)

    # Convert bits to +1/-1 (1 → 1, 0 → -1)
    pdm_signal = 2 * pdm_data.astype(np.float32) - 1

    # Step 2: Low-pass filter
    num_taps = 101  # length of FIR filter
    lpf = firwin(num_taps, cutoff=1.0/decimation_factor)
    filtered = lfilter(lpf, 1.0, pdm_signal)

    # Step 3: Decimate
    pcm = filtered[::decimation_factor]

    # Normalize to 16-bit PCM range
    pcm_norm = np.int16(pcm / np.max(np.abs(pcm)) * 32767)

    # Step 4: Save to WAV
    with wave.open(wav_file, 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(wav_sampling_rate)
        wf.writeframes(pcm_norm.tobytes())

    print("Conversion complete:", wav_file)
